In [1]:
# Task 1

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import shap

# Load the Titanic data
titanic = pd.read_csv("titanic_survival.csv")

titanic["age"] = titanic["age"].fillna(titanic["age"].median())

# The random forest uses five columns, same as the ensemble unit
rf_features = ["pclass", "age", "fare", "sibsp", "parch"]
X_rf = titanic[rf_features]
y = titanic["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X_rf, y, test_size=0.2, random_state=42
)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# The logistic regression uses two columns, same as the
# classification unit
log_features = ["pclass", "fare"]
X_log = titanic[log_features]

X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(
    X_log, y, test_size=0.2, random_state=42
)

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_log, y_train_log)

print("Random forest test accuracy:", rf_model.score(X_test, y_test))
print("Logistic regression test accuracy:",
      log_model.score(X_test_log, y_test_log))

Random forest test accuracy: 0.7262569832402235
Logistic regression test accuracy: 0.7039106145251397


In [2]:
# Task 2

coalition_values = {
    (): 0,
    ("A",): 10,
    ("B",): 4,
    ("C",): 6,
    ("A", "B"): 18,
    ("A", "C"): 19,
    ("B", "C"): 13,
    ("A", "B", "C"): 24,
}

def marginal(worker, before_group):
    after_group = tuple(sorted(before_group + (worker,)))
    before_group = tuple(sorted(before_group))
    return coalition_values[after_group] - coalition_values[before_group]

def average(values):
    return sum(values) / len(values)

A_contributions = [
    marginal("A", ()),
    marginal("A", ("B",)),
    marginal("A", ("C",)),
    marginal("A", ("B", "C")),
]

B_contributions = [
    marginal("B", ()),
    marginal("B", ("A",)),
    marginal("B", ("C",)),
    marginal("B", ("A", "C")),
]

C_contributions = [
    marginal("C", ()),
    marginal("C", ("A",)),
    marginal("C", ("B",)),
    marginal("C", ("A", "B")),
]

weights = [2, 1, 1, 2]

A_shapley = sum(c * w for c, w in zip(A_contributions, weights)) / 6
B_shapley = sum(c * w for c, w in zip(B_contributions, weights)) / 6
C_shapley = sum(c * w for c, w in zip(C_contributions, weights)) / 6

print(f"A Shapley: {A_shapley:.2f}")
print(f"B Shapley: {B_shapley:.2f}")
print(f"C Shapley: {C_shapley:.2f}")
print(f"Sum of values: {A_shapley + B_shapley + C_shapley}")

A Shapley: 11.50
B Shapley: 5.50
C Shapley: 7.00
Sum of values: 24.0


In [24]:
# Task 3

# Build explainer
explainer = shap.TreeExplainer(rf_model)
# Compute SHAP values for every row in the test set
shap_values = explainer(X_test)

print(shap_values.values.shape)

values = shap_values.values[0, :, 1]

for feature, value in zip(rf_features, values):
    print(f"{feature}: {value:.2f}")

(179, 5, 2)
pclass: -0.11
age: -0.05
fare: 0.03
sibsp: 0.03
parch: 0.04
